# Baseline Similarity Model (Cosine)

Baseline for audio similarity using spectrogram PNGs. 


## Setup
- full-only: `['full']`
- stems-only: `['bass', 'drums', 'vocals', 'other']`
- all 5: `['bass', 'drums', 'vocals', 'other', 'full']`

In [64]:
# Config (edit here)
IMAGE_SIZE = 32  # small for baseline speed; increase for higher fidelity
MAX_SONGS = None  # set a number for quick tests
CHANNELS = ['bass', 'drums', 'vocals', 'other', 'full']  # options: ['full'], stems-only, or all 5
TOPK = 10


In [65]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from typing import List


## Data Loading 

In [66]:
# Paths (relative to repo root)
META_PATH = Path('../data/metadata.pkl')
SPEC_ROOT = Path('../data/processed/specs')


In [67]:
# Path Checks
print("META_PATH =", META_PATH)
print("exists:", META_PATH.exists())

print("SPEC_ROOT =", SPEC_ROOT)
print("exists:", SPEC_ROOT.exists())

META_PATH = ../data/metadata.pkl
exists: True
SPEC_ROOT = ../data/processed/specs
exists: True


In [68]:
''' 
FIX!! 
Adding this is sloppy. Meet with group for how spectrogram folders will be handled.
'''

def normalize_song_id(name: str) -> str:
    """Normalize folder names like 'abc123 (1)' -> 'abc123'."""
    if name.endswith(' (1)'):
        return name[:-4]
    return name


In [69]:
def iter_song_dirs(spec_root: Path):
    for pkl in spec_root.glob('pkl_*'):
        for song_dir in pkl.iterdir():
            if song_dir.is_dir():
                yield song_dir


In [70]:
def build_spec_paths(spec_root: Path) -> pd.DataFrame:
    """
    Build a spec path table with defensive handling for:
    - duplicate folders like "<id> (1)"
    - incomplete folders missing required PNGs
    """
    rows = []
    seen = set()
    skipped_duplicate = 0
    skipped_incomplete = 0

    for song_dir in iter_song_dirs(spec_root):
        raw_id = song_dir.name
        song_id = normalize_song_id(raw_id)

        # Prefer the non-(1) folder if both exist
        if song_id in seen:
            skipped_duplicate += 1
            continue

        paths = {
            'spec_bass_path': song_dir / 'bass.png',
            'spec_drums_path': song_dir / 'drums.png',
            'spec_vocals_path': song_dir / 'vocals.png',
            'spec_other_path': song_dir / 'other.png',
            'spec_full_path': song_dir / f"{song_dir.name}.png",
        }

        # require all five channels for a clean baseline
        if not all(p.exists() for p in paths.values()):
            skipped_incomplete += 1
            continue

        rows.append({
            'spotify_id': song_id,
            **{k: str(v) for k, v in paths.items()},
        })
        seen.add(song_id)

    df = pd.DataFrame(rows)
    print('songs kept:', len(df))
    print('skipped duplicate dirs:', skipped_duplicate)
    print('skipped incomplete dirs:', skipped_incomplete)
    return df


In [71]:
def load_dataframe(meta_path: Path, spec_root: Path) -> pd.DataFrame:
    df_meta = pd.read_pickle(meta_path)
    df_spec_paths = build_spec_paths(spec_root)

    # inner join keeps only ids with clean spectrograms
    df = df_meta.merge(df_spec_paths, how='inner', on='spotify_id')
    return df.reset_index(drop=True)


In [72]:
df = load_dataframe(META_PATH, SPEC_ROOT)
print('rows after clean merge:', len(df))


songs kept: 9997
skipped duplicate dirs: 4
skipped incomplete dirs: 1
rows after clean merge: 9997


## Sanity Checks

In [73]:
# Quick sanity checks 
print('unique spotify_id:', df['spotify_id'].nunique())
print('rows:', len(df))
print('duplicate ids:', len(df) - df['spotify_id'].nunique())
print(df[['spotify_id','name','artist']].head())


unique spotify_id: 9997
rows: 9997
duplicate ids: 0
               spotify_id                              name  \
0  0dnz7bSs3txd9nGY9e3Mlf                         Limelight   
1  0nvIhBnscX9w7P2yrqxB6K           Friends Will Be Friends   
2  0a5wz1j3rnDa6IawNoZp5C            Tales of Brave Ulysses   
3  0zWtqc7q1JrB1gKyI501ph  I Heard It Through the Grapevine   
4  05JDbUa3kKV7sOwok4Xq4Z                 Gimme Three Steps   

                         artist  
0                          Rush  
1                         Queen  
2                         Cream  
3  Creedence Clearwater Revival  
4                Lynyrd Skynyrd  


## Choose Channels

Edit `CHANNELS` in the Config cell above. Examples:
- full-only: `['full']`
- stems-only: `['bass', 'drums', 'vocals', 'other']`
- all 5: `['bass', 'drums', 'vocals', 'other', 'full']`


In [74]:
CHANNEL_TO_COL = {
    'bass': 'spec_bass_path',
    'drums': 'spec_drums_path',
    'vocals': 'spec_vocals_path',
    'other': 'spec_other_path',
    'full': 'spec_full_path',
}

SELECTED_COLS = [CHANNEL_TO_COL[c] for c in CHANNELS]
print('using columns:', SELECTED_COLS)


using columns: ['spec_bass_path', 'spec_drums_path', 'spec_vocals_path', 'spec_other_path', 'spec_full_path']


## Baseline Embedding

In [75]:
def load_gray_resized(path: str, image_size: int) -> np.ndarray:
    img = Image.open(path).convert('L')
    img = img.resize((image_size, image_size), resample=Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr


In [76]:
def song_embedding(row: pd.Series, image_size: int, cols: List[str]) -> np.ndarray:
    chans = [load_gray_resized(row[c], image_size=image_size) for c in cols]
    x = np.stack(chans, axis=0)  # (C, H, W)
    return x.reshape(-1)


In [77]:
def l2_normalize(x: np.ndarray, eps: float = 1e-10) -> np.ndarray:
    n = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / (n + eps)


In [78]:
def build_embeddings(df: pd.DataFrame, image_size: int, cols: List[str], max_songs: int | None = None, log_every: int = 200) -> np.ndarray:
    n = len(df) if max_songs is None else min(len(df), max_songs)
    emb_list = []
    for i in range(n):
        if log_every and i % log_every == 0:
            print(f"embedding {i+1}/{n}")
        emb_list.append(song_embedding(df.iloc[i], image_size=image_size, cols=cols))
    emb = np.stack(emb_list, axis=0)
    return l2_normalize(emb)


## Build Embeddings

In [79]:
emb = build_embeddings(
    df,
    image_size=IMAGE_SIZE,
    cols=SELECTED_COLS,
    max_songs=MAX_SONGS,
    log_every=200, # Adjust if you want more/less updates
)
print('embeddings shape:', emb.shape)


embedding 1/9997
embedding 201/9997
embedding 401/9997
embedding 601/9997
embedding 801/9997
embedding 1001/9997
embedding 1201/9997
embedding 1401/9997
embedding 1601/9997
embedding 1801/9997
embedding 2001/9997
embedding 2201/9997
embedding 2401/9997
embedding 2601/9997
embedding 2801/9997
embedding 3001/9997
embedding 3201/9997
embedding 3401/9997
embedding 3601/9997
embedding 3801/9997
embedding 4001/9997
embedding 4201/9997
embedding 4401/9997
embedding 4601/9997
embedding 4801/9997
embedding 5001/9997
embedding 5201/9997
embedding 5401/9997
embedding 5601/9997
embedding 5801/9997
embedding 6001/9997
embedding 6201/9997
embedding 6401/9997
embedding 6601/9997
embedding 6801/9997
embedding 7001/9997
embedding 7201/9997
embedding 7401/9997
embedding 7601/9997
embedding 7801/9997
embedding 8001/9997
embedding 8201/9997
embedding 8401/9997
embedding 8601/9997
embedding 8801/9997
embedding 9001/9997
embedding 9201/9997
embedding 9401/9997
embedding 9601/9997
embedding 9801/9997
embeddi

## Retrieval (Cosine Similarity)

In [82]:
def topk_cosine(emb: np.ndarray, query: np.ndarray, k: int):
    if not np.isfinite(emb).all():
        raise ValueError('Embedding matrix contains NaN or inf values. Rebuild embeddings or restart the kernel.')
    if not np.isfinite(query).all():
        raise ValueError('Query vector contains NaN or inf values. Rebuild embeddings or restart the kernel.')
    # Use float64 einsum here to avoid spurious float32 matmul warnings from some NumPy/OpenBLAS builds.
    scores = np.einsum('ij,j->i', emb.astype(np.float64), query.astype(np.float64), optimize=True)
    if not np.isfinite(scores).all():
        raise ValueError('Cosine scores contain NaN or inf values after matmul.')
    k = int(min(max(k, 1), len(scores)))
    idx = np.argpartition(-scores, kth=k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return idx, scores[idx]


In [83]:
# Example query: choose a row index or a spotify_id
query_idx = 0
query_vec = emb[query_idx]
print('Embedding sanity:')
print('  emb dtype/shape:', emb.dtype, emb.shape)
print('  finite emb/query:', np.isfinite(emb).all(), np.isfinite(query_vec).all())
print('  norm range:', np.linalg.norm(emb, axis=1).min(), np.linalg.norm(emb, axis=1).max())
idx, scores = topk_cosine(emb, query_vec, k=TOPK)
print('  score range:', scores.min(), scores.max())


print('Query:')
print(df.iloc[query_idx][['spotify_id','name','artist']])

print('Top-k neighbors:')
for rank, (i, s) in enumerate(zip(idx, scores), start=1):
    row = df.iloc[int(i)]
    print(f"{rank:02d} score={s:.4f} id={row['spotify_id']} | {row['artist']} - {row['name']}")


Embedding sanity:
  emb dtype/shape: float32 (9997, 5120)
  finite emb/query: True True
  norm range: 0.9999999 1.0000001
  score range: 0.9575041955329954 0.99999998618278
Query:
spotify_id    0dnz7bSs3txd9nGY9e3Mlf
name                       Limelight
artist                          Rush
Name: 0, dtype: object
Top-k neighbors:
01 score=1.0000 id=0dnz7bSs3txd9nGY9e3Mlf | Rush - Limelight
02 score=0.9658 id=0tIJ9LcPpRVH2ENDYr1qjF | The Clash - Cheat
03 score=0.9603 id=5fsclfTnWFk4IWNkqTLEX4 | The National - Lit Up
04 score=0.9602 id=0byNlqYNXbwnYie5QH7Vtx | Bruce Springsteen - Human Touch
05 score=0.9598 id=1c29sP8Dp2hLl2w371nYpQ | Bad Religion - Flat Earth Society
06 score=0.9597 id=0taJtbIJrm1vDcz8r3IDjs | Gary Numan - Metal
07 score=0.9596 id=2fLiisZYOeVcHa5IGFLyxb | Radiohead - The Trickster
08 score=0.9591 id=3jpKxPNk2dFsODn64vk7YN | Riverside - Beyond The Eyelids
09 score=0.9577 id=0PTw1leQAEP1j4pdmczN0L | Slayer - Love To Hate
10 score=0.9575 id=7nFfCC5sBhwRVjdaciVGME | Misfits 

## Save Embeddings

In [85]:
SAVE_PATH = Path('../data/processed/baseline_embeddings.npz')
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

np.savez(
    SAVE_PATH,
    embeddings=emb.astype(np.float32),
    spotify_id=df['spotify_id'].to_numpy(),
    name=df['name'].to_numpy(),
    artist=df['artist'].to_numpy(),
    channels=np.array(CHANNELS, dtype=object),
    image_size=IMAGE_SIZE,
)

print('saved:', SAVE_PATH)


saved: ../data/processed/baseline_embeddings.npz
